In [1]:
from IPython.display import clear_output
from time import sleep
from datetime import datetime
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 100)
from bs4 import BeautifulSoup
import re
import sqlite3
from contextlib import closing # 
sqlite3.register_adapter(np.int64, lambda val: int(val)) # dataframe <-> sqlite3
from mycolor import *
print("library loading ", datetime.now().strftime("%Y/%m/%d %H:%M:%S"))

library loading  2019/11/02 20:36:50


In [2]:
# import importlib
# importlib.reload(mr)
import myrace as mr

In [7]:
""" スクレイピング用の関数 """
def use_urllib(url):
    import urllib.request
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 6.1; Win64; x64; rv:70.0) Gecko/20100101 Firefox/70.0"}
    req = urllib.request.Request(url, None, headers) # None->timeout=*
    try:
        res = urllib.request.urlopen(req)
    except urllib.error.URLError as e:
        if hasattr(e, 'reason'):
            print('We failed to reach a server.')
            print('Reason: ', e.reason)
        elif hasattr(e, 'code'): # HTTPError
            print('The server couldn\'t fulfill the request.')
            print('Error code: ', e.code)
    else:
        html = res.read()
        return html

def use_selenium(url):
    import warnings
    warnings.filterwarnings('ignore')
    # -> warn('Selenium support for PhantomJS has been deprecated, please use headless '
    from selenium import webdriver
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    # from selenium.webdriver.common.by import By
    from selenium.common.exceptions import TimeoutException
    
    driver = webdriver.PhantomJS()
    driver.get(url)
    for _ in range(3):
        try:
            WebDriverWait(driver, 15).until(EC.presence_of_all_elements_located)
            # WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.ID, 'race_main')))
        except TimeoutException as te:
            print("Web driver: ", te)
        else:
            break
    else:
            driver.close()
            sys.exit()
    html = driver.page_source
    driver.close()
    return html
print("Scraping Function loaded ", datetime.now().strftime("%Y/%m/%d %H:%M:%S"))

Scraping Function loaded  2019/11/02 20:54:51


In [3]:
class RaceScraping():
    """ レースデータを取得するクラス """  
    def __init__(self, RaceCode):
        """ 初期化、基本値の取得 """
        self.RaceCode = RaceCode
        url = "https://race.netkeiba.com/?pid=race_old&id=c" + self.RaceCode
        html = use_urllib(url)
        soup = BeautifulSoup(html,"lxml")
        self.soup = soup
        self.Date = soup.find("div", class_="race_otherdata").find_all("p")[0].string
        self.RaceName = re.sub("\n","", soup.find("title").string.split(" ")[0])
        _course = soup.find("div", class_="race_otherdata").find_all("p")[1]
        self.Course = re.sub("<p>|\d+回|\d+日目","", str(_course).split("\xa0")[0])
        self.R = soup.find("dl", class_="racedata fc").find("dt").string.strip("\n")
        horse_tags = soup.find_all("td", class_="txt_l horsename")
        self.horse_names = [tag.find("a").string for tag in horse_tags]
        self.horse_links = [tag.find("a").get("href") for tag in horse_tags]
        self.horse_codes = [re.search(r"\d+", link).group() for link in self.horse_links]
        print(f"Scraped {self.Course}{self.R} {self.RaceName}'s page.")
    
    def get_race(self):
        """ レース """
        _trkdist = self.soup.find("dl", class_="racedata fc").find_all("p")[0].string
        TrkDist = re.sub("\xa0"," ", _trkdist)
        _cond = self.soup.find("dl", class_="racedata fc").find_all("p")[1].string
        Cond = re.sub("\xa0"," ", _cond)
        _holding = self.soup.find("div", class_="race_otherdata").find_all("p")[1].string
        Holding = re.sub("\xa0"," ", _holding)
        _detail = self.soup.find("div", class_="race_otherdata").find_all("p")[2].string
        Detail = re.sub("\xa0"," ", _detail)
        _prize = self.soup.find("div", class_="race_otherdata").find_all("p")[3].string
        Prize =  re.sub("\xa0"," ", _prize) 
        col = ('RaceCode', 'Date', 'Course', 'R', 'RaceName', 'TrkDist', 'Cond',\
               'Holding', 'Detail', 'Prize')
        tpl = (self.RaceCode, self.Date, self.Course, self.R, self.RaceName,\
               TrkDist, Cond, Holding, Detail, Prize)
        race_sr = pd.Series(tpl, col, name='race')
        print(f"Loading {self.Course}{self.R} {self.RaceName}'s race.")
        return race_sr
    
    def get_entries(self):
        """ 出馬表 """
        df = pd.io.html.read_html(self.soup.prettify())
        entries_df = df[0].copy()
        entries_df.columns = entries_df.columns.droplevel()
        entries_df.fillna('', inplace=True) # nanを''（空白）文字列に置き換える
        horses = [row[1:] for row in entries_df.itertuples(name=None)]
        col = ('RaceCode', 'Bk', 'Hs', 'Horse', 'SexAge', 'Weight',\
               'Jockey', 'Trainer' ,'HsWeight', 'Odds', 'Fav', 'HorseCode') 
        entries_lst=[]
        for i, horse in enumerate(horses):
            tpl = (self.RaceCode,) + horse + (self.horse_codes[i],)
            horse_sr = pd.Series(tpl, col, name='entry')
            entries_lst.append(horse_sr)
        print(f"Loading {self.Course}{self.R} {self.RaceName}'s Entries.")
        return entries_lst
       
    def get_horses(self):
        """ 競走馬、過去のレース """
        h_col = ('RaceCode', 'HorseCode', 'Horse', 'Birthday', 'Trainer', 'Breeder',\
           'Owner', 'Producing', 'MarketPrice', 'Prize', 'TotalResult', 'MainVictory',\
           'CloseRelative', 'Sire', 'Dam', 'SiresSire', 'SiresDam', 'DamsSire', 'DamsDam')
        p_col = ('RaceCode','HorseCode','Horse','Date','Holding','Weather','R','RaceName',\
               'Entry','Bk','Hs','Odds','Fav','FP','Jockey','Weight','TrkDist','Cond',\
               'Finish','Margin','Passing','Pace','Last3h','HsWeight','FirstHorse','Prize')
        empty_df = pd.DataFrame([], columns=p_col)
        horse_lst = []
        for HorseCode, Horse, url in zip(self.horse_codes, self.horse_names, self.horse_links):
            html = use_urllib(url)
            soup = BeautifulSoup(html,"lxml")
            df = pd.io.html.read_html(soup.prettify())
            """ ower's infomation """
            owner_df = df[1].copy()
            owner_df.fillna('', inplace=True) # nanを空欄へ変換
            dic = dict([(i, v) for i, v in zip(owner_df[0], owner_df[1])])
            Birthday = dic['生年月日']
            Trainer = dic['調教師']
            Owner = dic['馬主']
            Breeder = dic['生産者']
            Producing = dic['産地']
            MarketPrice = dic['セリ取引価格']
            Prize = dic['獲得賞金']
            TotalResult = dic['通算成績']
            MainVictory = dic['主な勝鞍']
            CloseRelative = dic['近親馬']
            """ pedigree """
            pedigree_df = df[2].copy()
            Sire = pedigree_df[0][0]
            Dam = pedigree_df[0][2]
            SiresSire = pedigree_df[1][0]
            SiresDam = pedigree_df[1][1]
            DamsSire = pedigree_df[1][2]
            DamsDam = pedigree_df[1][3]
            tpl = (self.RaceCode, HorseCode, Horse, Birthday, Trainer, Breeder,\
                 Owner, Producing, MarketPrice, Prize, TotalResult, MainVictory,\
                 CloseRelative, Sire, Dam, SiresSire, SiresDam, DamsSire, DamsDam)            
            horse_sr = pd.Series(tpl, h_col, name='horse')
            
            """ result 過去のレース """
            flg = False
            for d in df:
                if('日付' in d.columns):
                    temp_df = d.copy()
                    del temp_df['映  像']
                    del temp_df['馬場  指数']
                    del temp_df['ﾀｲﾑ  指数']
                    del temp_df['厩舎  ｺﾒﾝﾄ']
                    del temp_df['備考']
                    temp_df['賞金'].fillna('', inplace=True)
                    temp_df.dropna(inplace=True) # nanを含む行を削除
                    racedate = datetime.strptime(self.Date[0:10], '%Y/%m/%d')
                    tpls = []
                    for td in temp_df.itertuples(name=None):
                        pastdate = datetime.strptime(td[1],'%Y/%m/%d')
                        if(pastdate < racedate):
                            tpls.append(tuple(td[1:]))
                    if(len(tpls)!=0):
                        p_tpls = [(self.RaceCode, HorseCode, Horse) + tpl for tpl in tpls]
                        performance_df = pd.DataFrame(p_tpls, columns=p_col)
                        horse_data = [horse_sr, performance_df]
                        flg = True
            if(flg==False):
                horse_data = [horse_sr, empty_df]
            horse_lst.append(horse_data)
        print(f"Scraped and Loading {self.Course}{self.R} {self.RaceName}'s Horses.")
        return horse_lst
    
    def get_horsepillar(self):
        race = self.get_race()
        entries = self.get_entries()
        horses = self.get_horses()
        horsepillar = []
        for entry, horse in zip(entries, horses):
            horsepillar.append(pd.Series([race, entry, horse[0], horse[1]],\
                            ['Race', 'Entry', 'Horse', 'Performance'], name='horsepiller'))
        print(f"Scraped and Loading {self.Course} {self.R} {self.RaceName}'s Horsepiller.")
        return horsepillar

class ResultScraping():
    """ レース結果を取得するクラス """ 
    def __init__(self, RaceCode):
        """ 初期化、基本値の取得 """
        self.RaceCode = RaceCode
        url = "https://db.netkeiba.com/race/" + self.RaceCode
        html = use_urllib(url)
        soup = BeautifulSoup(html,"lxml")
        found = soup.find("p", class_='smalltxt')
        if found is None:
            print('No infomation on the page.')
            return
        self.Course = re.sub("\d+回|\d+日目","", \
                             soup.find("p", class_='smalltxt').string.split(" ")[1])
        self.R = soup.find("dl", class_="racedata fc").find("dt").string\
                            .strip("\n").replace(" ", "")
        self.RaceName = soup.find("title").string.split(" ")[0].split("｜")[0]
        result_html = soup.find_all("table")[0].prettify()
        self.result_df = pd.io.html.read_html(result_html)[0]
        pay1_df = pd.io.html.read_html(soup.find_all("table")[1].prettify())[0]
        pay2_df = pd.io.html.read_html(soup.find_all("table")[2].prettify())[0]
        self.payout_df = pd.concat([pay1_df, pay2_df])
        print(f"Scraped {self.Course}{self.R} {self.RaceName}'s result and payout.")
    
    def get_result(self):
        """ result レース結果 """
        _result_df = self.result_df.copy()
        _result_df.fillna('', inplace=True)
        data = [(rc,) + row[1:] for row in _result_df.itertuples(name=None)]
        idx = ('RaceCode', 'FP', 'Bk', 'Hs', 'Horse', 'SexAge', 'Weight', 'Jockey', \
            'Finish', 'Margin', 'Fav', 'odds', 'Trainer', 'HsWeight')
        result_lst = [pd.Series(d, idx) for d in data]
        return result_lst

    def _procpayout(self):
        """ payout 払い戻し表の加工 """
        def process_payout(series_lst, process_lst=[]): # 払い出しSeriesの加工
            if(len(series_lst)==0):
                return process_lst
            s = series_lst.pop(0)
            if(s.name=='複勝' or s.name=='ワイド'):
                placed = s['Placed'].split("  ")
                payout = s['Payout'].split("  ")
                fav = s['Fav'].split("  ")
                rng = range(len(placed))
                data = [[placed[i], payout[i], fav[i]] for i in rng]
                process_lst += [pd.Series(data[i], index=idx, name=f"{s.name}{i+1}") for i in rng]
            else:
                process_lst.append(s)
            return process_payout(series_lst)

        temp_df = self.payout_df.copy()
        series_lst = []
        idx = ['Placed', 'Payout', 'Fav']
        for row in temp_df.itertuples(name=None):
            sr = (pd.Series(row[2:], index=idx, name=row[1]))
            series_lst.append(sr)
        proc_lst = process_payout(series_lst)
        return  proc_lst
    
    def get_payout(self):
        """ payout 払い戻し """
        proc_lst = self._procpayout()
        data = [(rc, p.name, p['Placed'], p['Payout'], p['Fav']) for p in proc_lst]
        idx = ['RaceCode', 'Ticket', 'Placed', 'Payout', 'Fav']
        payout_lst = [pd.Series(d, idx) for d in data]
        return payout_lst
    
    def get_raceresult(self):
        """ 結果と払い戻し """
        result = self.get_result()
        payout = self.get_payout()
        return result, payout
    
    def show_racename(self):
        return print(self.Course, self.R, self.RaceName)
    
    def show_result(self):
        result_lst = self.get_result()
        _result_df = pd.DataFrame(result_lst)
        result_df = _result_df.drop('RaceCode', axis=1)
        return display(result_df)
    
    def show_payout(self):
        proc_lst = self._procpayout()
        payout_df = pd.DataFrame(proc_lst).style.set_properties(**{'text-align': 'right'})
        return display(payout_df)
    
    def show_all(self):
        self.show_racename()
        self.show_result()
        self.show_payout()
    
print("Scraping class loaded ", datetime.now().strftime("%Y/%m/%d %H:%M:%S"))

Scraping class loaded  2019/11/02 20:53:52


In [4]:
""" レースデータをsqlite3に登録するクラス """
class Registration():
    def __init__(self):
        pass
    
    def execute_one(self, sql, data):
        """ sqlite3へ登録（単一レコード） """
        with closing(sqlite3.connect("racing.sqlite3")) as conn:
            cur = conn.cursor()
            try:
                cur.execute(sql, data)
                conn.commit()
            except sqlite3.Error as e:
                print('sqlite3.Error occurred: ', e.args[0])
            conn.close()
    
    def execute_many(self, sql, data):
        """ sqlite3へ登録(複数レコード) """
        with closing(sqlite3.connect("racing.sqlite3")) as conn:
            cur = conn.cursor()
            try:
                cur.executemany(sql, data)
                conn.commit()
            except sqlite3.Error as e:
                print('sqlite3.Error occurred: ', e.args[0])
            conn.close()
    
    def insert_race(self, race_sr):
        """  race レース情報の登録"""
        data = tuple(race_sr.values)
        sql = f"Insert into Race {str(tuple(race_sr.index.values))}\
            values(?,?,?,?,?,?,?,?,?,?)"
        self.execute_one(sql, data)
        print("Race data entred.")
        
    def replace_race(self, race_sr):
        """  race レース情報の再登録"""
        data = tuple(race_sr.values)
        sql = f"Replace into Race {str(tuple(race_sr.index.values))}\
            values(?,?,?,?,?,?,?,?,?,?)"
        self.execute_one(sql, data)
        print("Race data replaced.")
    
    def insert_entries(self, entry_lst):
        """ entries 出馬表の登録 """
        data = [tuple(e.values) for e in entry_lst]
        sql = f"Insert into Entries {str(tuple(entry_lst[0].index.values))}\
            Values(?,?,?,?,?,?,?,?,?,?,?,?)"
        self.execute_many(sql, data)
        print("Entries data entred.")
    
    def insert_horses(self, horse_lst):
        """ horses 競争馬の登録 """
        horses = [horse[0] for horse in horse_lst]
        data = [tuple(h.values) for h in horses]
        sql = f"Insert into Horse {str(tuple(horses[0].index.values))}\
            Values(?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)"
        self.execute_many(sql, data)
        print("Horse data entred.")
    
    def insert_performance(self, horse_lst):
        """ performance 競争成績の登録 """
        df_lst = [horse[1] for horse in horse_lst]
        sql = f"Insert into Performance {str(tuple([d for d in df_lst[0]]))}\
            Values(?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)"
        flg = False
        for df in df_lst:
            data = [td[1:] for td in df.itertuples(name=None)]
            if(len(data)!=0):
                self.execute_many(sql, data)
                flg = True
        if(flg==True): print("Performance data entred.")
        else: print("Performance data is nothing.")
    
    def register_horsepillar(self, horsepillar):
        race_sr = horsepillar[0]['Race']
        entry_lst = [horse['Entry'] for horse in horsepillar]
        horse_lst = [(horse['Horse'], horse['Performance']) for horse in horsepillar]
        self.insert_race(race_sr)
        self.insert_entries(entry_lst)
        self.insert_horses(horse_lst)
        self.insert_performance(horse_lst)
        
    def insert_result(self, result_lst):
        data = [tuple(r.values) for r in result_lst]
        sql = f"INSERT INTO Result {str(tuple(result_lst[0].index.values))} \
            Values(?,?,?,?,?,?,?,?,?,?,?,?,?,?)"
        self.execute_many(sql, data)
        print("Result data entred.")
        
    def insert_payout(self, payout_lst):
        data = [tuple(p.values) for p in payout_lst]
        sql = f"INSERT INTO Payout {str(tuple(payout_lst[0].index.values))} \
            Values(?,?,?,?,?)"
        self.execute_many(sql, data)
        print("Payout data entred.")
        
    def register_raceresult(self, result, payout):
        self.insert_result(result)
        self.insert_payout(payout)
            
print("Registration class loaded ", datetime.now().strftime("%Y/%m/%d %H:%M:%S"))

Registration class loaded  2019/11/02 20:53:58


In [29]:
"""
sqlite3 
"""
def sql3_data(table, rc):
    with closing(sqlite3.connect("racing.sqlite3")) as conn:
        cur = conn.cursor()
        sql = f"Select * from {table} Where RaceCode = ?"
        try:
            cur.execute(sql ,(rc,))
        except sqlite3.Error as e:
            print('sqlite3.Error occurred:', e.args[0])
        res =  cur.fetchall()
        col = list(map(lambda x: x[0], cur.description))
        conn.commit()
        conn.close()
    return res, col
    try:
        cur.execute(Create_Table)
    except sqlite3.Error as e:
        print('sqlite3.Error occurred:', e.args[0])
    conn.commit()
    conn.close()

In [40]:
rc = mr.rc('京都sun') + '01'
pf = sql3_data('Performance', rc)
lst = pf[0]
[(l[3], l[4], l[5], l[6], l[8], l[9], l[10], l[11], l[12], l[13], l[14], l[15])for l in lst]

[('キクノヴィゴ',
  '2019/10/26',
  '4京都8',
  '曇',
  '2歳未勝利',
  14,
  1,
  1,
  168.4,
  13,
  12,
  '松岡正海'),
 ('キクノヴィゴ', '2019/10/12', '4京都3', '雨', '2歳新馬', 16, 2, 4, 67.8, 15, 14, '松田大作'),
 ('トウカイバリエンテ',
  '2019/10/19',
  '4京都6',
  '曇',
  '2歳未勝利',
  16,
  1,
  1,
  105.2,
  11,
  8,
  '亀田温心'),
 ('トウカイバリエンテ',
  '2019/10/06',
  '4京都2',
  '晴',
  '2歳新馬',
  16,
  3,
  6,
  53.8,
  10,
  13,
  '亀田温心'),
 ('タガノウィリアム',
  '2019/10/13',
  '4京都4',
  '曇',
  '2歳未勝利',
  11,
  7,
  9,
  13.9,
  3,
  8,
  '松田大作'),
 ('タガノウィリアム',
  '2019/07/20',
  '3中京7',
  '曇',
  '2歳新馬',
  13,
  8,
  13,
  120.4,
  13,
  11,
  '川須栄彦'),
 ('ラブミーアーサー',
  '2019/10/20',
  '4京都7',
  '晴',
  '2歳未勝利',
  12,
  5,
  6,
  307.0,
  12,
  6,
  '柴田大知'),
 ('ラブミーアーサー',
  '2019/09/07',
  '4阪神1',
  '晴',
  '2歳新馬',
  14,
  5,
  7,
  275.2,
  12,
  14,
  '服部寿希'),
 ('シゲルカセイ', '2019/10/12', '4京都3', '雨', '2歳未勝利', 12, 1, 1, 3.1, 1, 3, '酒井学'),
 ('シゲルカセイ', '2019/09/21', '4阪神6', '曇', '2歳未勝利', 9, 2, 2, 9.0, 4, 2, '酒井学'),
 ('シゲルカセイ', '2019/09/01', '2小倉12'

In [ ]:
"""
data entry test (delet)
"""
def delete_onerace(rc):
    with closing(sqlite3.connect("racing.sqlite3")) as conn:
        cur = conn.cursor()
        for tbl in ['Race', 'Entries', 'Horse', 'Performance']:
            sql = f"Delete From {tbl} Where RaceCode = ?"
            cur.execute(sql ,(rc,))
            conn.commit()
            print(f"Delete: {rc} {tbl}")
        conn.close()

rc = mr.rc('福島sun') + '04'#str(i).rjust(2,"0")
delete_onerace(rc)

In [8]:
""" 1日分の登録 """
rgs = Registration()
for i in range(1,4):
    rc = mr.rc('京都sun') + str(i).rjust(2,"0")
    rcs = RaceScraping(rc)
    horsepillar = rcs.get_horsepillar()
    rgs.register_horsepillar(horsepillar)
#     rts = ResultScraping(rc)
#     result, payout = rts.get_raceresult()
#     rgs.register_raceresult(result, payout)
    sleep(1.5)
    clear_output()

In [ ]:
"""
data entry test
"""
rc = mr.grc('test') + '11'
# """ スクレイプ """
# rcs = RaceScraping(rc)
# race = rcs.get_race()
# entries = rcs.get_entries()
# horses = rcs.get_horses()
# horsepillar = rcs.get_horsepillar()
# """ 登録 """
# rgs = Registration()
# rgs.insert_race(race)
# rgs.replace_race(race)
# rgs.insert_entries(entries)
# rgs.insert_horses(horses)
# rgs.insert_performance(horses)
# rgs.register_horsepillar(horsepillar)
# """ 結果 """
# rts = ResultScraping(rc)
# result, payout = rts.get_raceresult()
# rgs.register_raceresult(result, payout)

In [ ]:
"""
data entry test2
"""
rc = mr.grc('test') + '11'
rsc = ResultScraping(rc)
rsc.show_all()

In [ ]:
class LeadingRegister():
    """ リーディングの登録 """
    def __init__(self):
        pass
        
    def to_df(self, url):
        html = use_urllib(url)
        soup = BeautifulSoup(html,"lxml")
        df = pd.io.html.read_html(soup.prettify())
        return df
    
    def get_jockey(self):
        lst=[]
        for i in range(1,5):
            url="https://db.netkeiba.com/?pid=jockey_leading&year=2019&page=" + str(i)
            _df = self.to_df(url)
            df = pd.concat([_df[0]['順位'], _df[0]['騎手名'], _df[0]['複勝  率']], axis=1)
            lst.append(df)
        jockey_df= pd.concat(lst, axis=0)
        dic ={'Jockey': jockey_df}
        return dic
    
    def get_trainer(self):
        lst=[]
        for i in range(1,6):
            url="https://db.netkeiba.com/?pid=trainer_leading&year=2019&page=" + str(i)
            _df = self.to_df(url)
            df = pd.concat([_df[0]['順位'], _df[0]['調教師名'], _df[0]['複勝  率']], axis=1)
            lst.append(df)
        trainer_df = pd.concat(lst, axis=0)
        dic = {'Trainer': trainer_df}
        return dic
        
    def get_sire(self):
        lst=[]
        for i in range(1,10):
            url="https://db.netkeiba.com/?pid=sire_leading&year=2019&page=" + str(i)
            _df = self.to_df(url)
            df = pd.concat([_df[0]['順位'], _df[0]['馬名'], _df[0]['勝馬  率']], axis=1)
            lst.append(df)
        sire_df = pd.concat(lst, axis=0)
        dic = {'Sire': sire_df}
        return dic
    
    def get_breeder(self):
        lst=[]
        for i in range(1,22):
            url="https://db.netkeiba.com/?pid=breeder_leading&year=2019&page=" + str(i)
            _df = self.to_df(url)
            df = pd.concat([_df[0]['順位'], _df[0]['生産者名'], _df[0]['複勝  率']], axis=1)
            lst.append(df)
        breeder_df = pd.concat(lst, axis=0)
        dic = {'Breeder': breeder_df}
        return dic
    
    def _execute_many(self, sql, data):
        """ sqlite3へ登録(複数レコード) """
        with closing(sqlite3.connect("racing.sqlite3")) as conn:
            cur = conn.cursor()
            try:
                cur.executemany(sql, data)
                conn.commit()
            except sqlite3.Error as e:
                print('sqlite3.Error occurred: ', e.args[0])
            conn.close()
        
    def insert_leading(self, dic):
        key = list(dic.keys())[0]
        data = [d[1:] for d in dic[key].itertuples()]
        sql = f"INSERT INTO {key}('Rank', 'Name', 'Rate') Values(?,?,?)"
        self._execute_many(sql, data)
        print(f"{key} leading data entred.")
    
    def usage(self):
        print("ld = LeadingRegister()")
        print("dic = ld.get_jockey()")
        print("ld.insert_leading(dic)")
        print("-> Jockey leading data entred.")

In [ ]:
ld = LeadingRegister()
ld.usage()

In [ ]:
""" 登録データの確認 """
tbl = 'Breeder'
with closing(sqlite3.connect("racing.sqlite3")) as conn:
    cur = conn.cursor()
    sql = f"Select * From {tbl};"
    cur.execute(sql)
    res = cur.fetchall()
    print(f"data: {len(res)}")
    for i, r in enumerate(res):
        if i < 10:
            print(r[1], r[2], r[3])
    conn.close()

In [ ]:
""" sqlite3から馬柱を作成する関数 rc <- RaceCode """
def set_horsepillar(rc):
    # table -> Race, Entries, Horse, Performance
    def fetch_race(rc):
        with closing(sqlite3.connect("racing.sqlite3")) as conn:
            cur = conn.cursor()
            sql = f"Select * from Race Where RaceCode = ?"
            cur.execute(sql, (rc,))
            res =  cur.fetchone()
            col = list(map(lambda x: x[0], cur.description))
            conn.close()
        return res, col
    
    def fetch_entries(rc):
        with closing(sqlite3.connect("racing.sqlite3")) as conn:
            cur = conn.cursor()
            sql = f"Select * from Entries Where RaceCode = ?"
            cur.execute(sql, (rc,))
            res =  cur.fetchall()
            col = list(map(lambda x: x[0], cur.description))
            conn.close()
        return res, col
    
    def fetch_horses(rc, horsecode, table): # Horse, Performance
        with closing(sqlite3.connect("racing.sqlite3")) as conn:
            cur = conn.cursor()
            sql = f"Select * from {table} Where RaceCode = ? and HorseCode = ?"
            cur.execute(sql, (rc, horsecode))
            res =  cur.fetchall()
            col = list(map(lambda x: x[0], cur.description))
            conn.close()
        return res, col
    
    race, r_col = fetch_race(rc)
    race_sr = pd.Series(race, index=r_col, name='race')
    
    entries, e_col = fetch_entries(rc)
    entry_lst = [pd.Series(entry, index=e_col, name='entries') for entry in entries]
    
    horse_lst=[]
    for entry in entry_lst:
        horsecode = entry['HorseCode']
        horse, h_col = fetch_horses(rc, horsecode, "Horse")
        horse_lst.append(pd.Series(horse[0], index=h_col, name='horse'))
    
    horsepillar=[]
    for entry, horse in zip(entry_lst, horse_lst):
        horsecode = horse['HorseCode']
        performance, p_col = fetch_horses(rc, horsecode, "Performance")
        if(len(performance) != 0):
            performance_df = pd.DataFrame(performance, columns=p_col)
        else:
            performance_df = pd.DataFrame([], columns=p_col)
        
        horse_sr = pd.Series([race_sr, entry, horse, performance_df], \
                             index=['Race', 'Entry', 'Horse', 'Performance'])
        horsepillar.append(horse_sr)
            
    return horsepillar

horsepillar = set_horsepillar(mr.grc('test') + '11')